# Anchoring classifier — validation against RNN ground truth

Every anchoring result in this project rests on the trial-correlation classifier being
right. On recorded data there is no ground truth, so the only place accuracy can be
*measured* rather than inferred is the simulated network.

**The ground truth.** `net7_1D_track_simulation.ipynb` runs one continuous session in
which the position cue is present for 35 trials (condition **A**), removed for 30 trials
(condition **C** — teleport still occurs but blind, so the hidden state carries over
uncorrected and the grid representation drifts off the track reference frame), then
restored for 35 trials. `phase_of_trial` is therefore a true per-trial anchored /
non-anchored label, and the classifier never sees it.

**What this notebook tests**

| panel | question |
|---|---|
| A | what do a hit and a miss actually look like? |
| B | accuracy and confusion across all units — the headline number, with its spread |
| C | does the phase-aliasing median filter earn its place? |
| D | how brief an epoch can the classifier resolve? |
| E | is it general, or does accuracy depend on how grid-like the unit is? |

Panel E is the one that matters for the claim made in the Methods. The classifier was
adopted over the spectral classifier because it was purpose-built to be general rather
than tuned on grid cells. If accuracy falls off for low-gridness units, that claim is
weaker than stated and should be softened.


In [ ]:
import os, sys, json, warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, median_filter
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, confusion_matrix
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Arial'

SIM_DIR = '/Users/harryclark/Documents/spatial-manifolds/data/rnn_xgboost/net7_1D_track_sim'
FIGPATH = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/AnchorDynamics2026'

_sw = np.load(os.path.join(SIM_DIR, 'switch_cache.npz'), allow_pickle=True)
_sim = np.load(os.path.join(SIM_DIR, 'sim_cache.npz'), allow_pickle=True)

TC = _sw['all_unit_tc_switch']              # (n_units, n_trials, n_bins)
PHASE = _sw['phase_of_trial'].astype(str)   # 'A' (cued/anchored) or 'C' (blind/non-anchored)
DECODE_ERR = _sw['decode_errs_switch']
GRID_SCORES = _sim['grid_scores']
BS_CM, TRACK_CM = float(_sim['bs']), float(_sim['tl'])

TRUTH = (PHASE == 'A').astype(int)          # 1 = anchored, 0 = non-anchored
N_UNITS, N_TRIALS, N_BINS = TC.shape
T1 = int(np.argmax(PHASE == 'C'))                       # cue lost
T2 = T1 + int((PHASE == 'C').sum())                     # cue restored

GC_COLOR, NGS_COLOR = '#c04744', '#2171b5'
# anchored / non-anchored colours, matching the rest of AnchorDynamics2026
from matplotlib.colors import ListedColormap, BoundaryNorm
ANCH_COLOR, NONANCH_COLOR = '#9d5391', '#5f9ea0'
TA_CMAP = ListedColormap([NONANCH_COLOR, ANCH_COLOR])
TA_NORM = BoundaryNorm([-0.5, 0.5, 1.5], TA_CMAP.N)

print(f'{N_UNITS} units x {N_TRIALS} trials x {N_BINS} bins ({BS_CM:.0f}cm bins, {TRACK_CM:.0f}cm track)')
print(f'ground truth: A[0:{T1}] -> C[{T1}:{T2}] -> A[{T2}:{N_TRIALS}]  '
      f'({int(TRUTH.sum())} anchored / {int((1-TRUTH).sum())} non-anchored trials)')
print(f'grid score range {GRID_SCORES.min():.2f} .. {GRID_SCORES.max():.2f}')

In [ ]:
import sys
sys.path.insert(0, '/Users/harryclark/Documents/spatial-manifolds/src')
# single source of truth for the classifier -- see Methods
from spatial_manifolds.anchoring import (trial_cluster_labels, smooth_nanaware,
                                        anchoring_cmap, ANCH_COLOR, NONANCH_COLOR)

def classify_all(tc, truth, medfilt_size=5, nan_aware=True):
    """-> labels (n_units, n_trials), accuracy, silhouette, per-unit."""
    lab = np.full(tc.shape[:2], np.nan)
    sil = np.full(tc.shape[0], np.nan)
    for u in range(tc.shape[0]):
        l, _, _, s = trial_cluster_labels(tc[u], medfilt_size=medfilt_size,
                                          nan_aware=nan_aware)
        lab[u], sil[u] = l, s
    ok = ~np.isnan(lab).any(axis=1)
    acc = np.full(tc.shape[0], np.nan)
    acc[ok] = (lab[ok] == truth[None, :]).mean(axis=1)
    return lab, acc, sil, ok


import time; _t = time.time()
LAB, ACC, SIL, OK = classify_all(TC, TRUTH, medfilt_size=5, nan_aware=True)
LAB_Z, ACC_Z, _, OK_Z = classify_all(TC, TRUTH, medfilt_size=5, nan_aware=False)
print(f'zero-filled (original): median accuracy {np.nanmedian(ACC_Z):.3f}')
print(f'NaN-aware smoothing   : median accuracy {np.nanmedian(ACC):.3f}')
print(f'classified {int(OK.sum())}/{N_UNITS} units in {time.time()-_t:.0f}s')
print(f'accuracy: median {np.nanmedian(ACC):.3f}, mean {np.nanmean(ACC):.3f}, '
      f'IQR [{np.nanpercentile(ACC,25):.3f}, {np.nanpercentile(ACC,75):.3f}]')
print(f'units >= 0.90 accuracy: {int((ACC >= 0.90).sum())}/{int(OK.sum())} '
      f'({100*np.nanmean(ACC >= 0.90):.1f}%)')
print(f'units <= 0.50 (at or below chance): {int((ACC <= 0.50).sum())}')

## Panel A — what a hit and a miss look like

Two units with high accuracy and one with low, each shown as the trial x position rate
map with the true and inferred label sequences drawn underneath. The dashed lines mark
cue loss and cue restoration.

In [ ]:
def draw_labels(ax, seq, ylabel):
    img = np.asarray(seq, float)[None, :]
    ax.imshow(img, aspect='auto', interpolation='nearest', cmap=TA_CMAP, norm=TA_NORM)
    ax.set_yticks([]); ax.set_xticks([])
    ax.set_ylabel(ylabel, fontsize=7, rotation=0, ha='right', va='center')

order = np.argsort(np.where(np.isnan(ACC), -1, ACC))
examples = [order[-1], order[-2], order[int(0.02*len(order))]]   # 2 best, 1 poor
titles = ['best', '2nd best', 'poor (2nd percentile)']

fig, axes = plt.subplots(3, 3, figsize=(10, 5.0),
                         gridspec_kw={'height_ratios': [6, 1, 1], 'hspace': 0.12, 'wspace': 0.18})
for col, (u, tt) in enumerate(zip(examples, titles)):
    ax = axes[0, col]
    # Show the map as the classifier sees it: NaN-aware smoothed. The raw map is
    # unreadable here for two separate reasons -- 22% of position bins are
    # unvisited on a given trial (true gaps), and ~70% of the bins that WERE
    # visited contain no spikes (real zeros, this unit fires sparsely). Plotting
    # raw counts renders both as dark speckle and hides the spatial structure.
    smoothed = np.array([smooth_nanaware(TC[u][t].astype(float), 1.5)
                         for t in range(N_TRIALS)])
    cmap = plt.get_cmap('viridis').copy(); cmap.set_bad('0.85')
    ax.imshow(np.ma.masked_invalid(smoothed), aspect='auto', interpolation='nearest',
              cmap=cmap, extent=[0, TRACK_CM, N_TRIALS, 0])
    for t, c in ((T1, 'tab:red'), (T2, 'tab:green')):
        ax.axhline(t, color=c, ls='--', lw=0.9)
    ax.set_title(f'unit {u} — {tt}\naccuracy {ACC[u]:.2f}, grid score {GRID_SCORES[u]:.2f}', fontsize=8)
    ax.set_xlabel('position (cm)', fontsize=8)
    if col == 0: ax.set_ylabel('trial', fontsize=8)
    draw_labels(axes[1, col], TRUTH, 'true')
    draw_labels(axes[2, col], LAB[u], 'inferred')
    wrong = np.where(LAB[u] != TRUTH)[0]
    for w in wrong:
        axes[2, col].plot(w, 0, marker='v', color='tab:red', markersize=3, clip_on=False)
    axes[2, col].set_xlabel('trial', fontsize=8)
fig.suptitle('Anchoring classifier vs RNN ground truth — rate maps NaN-aware smoothed as the classifier sees them;\nmauve = anchored, pale = non-anchored; red triangles = misclassified trials', fontsize=9)
plt.savefig(os.path.join(FIGPATH, 'anchoring_validation_A_examples.pdf'),
            dpi=200, bbox_inches='tight')
plt.show()

## Panel B — accuracy and confusion across all units

In [ ]:
cm = confusion_matrix(np.tile(TRUTH, int(OK.sum())),
                      LAB[OK].ravel().astype(int), labels=[0, 1])
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))

ax = axes[0]
ax.hist(ACC[OK], bins=np.linspace(0, 1, 41), color='0.4')
ax.axvline(np.nanmedian(ACC), color='tab:red', lw=1.2,
           label=f'median {np.nanmedian(ACC):.3f}')
ax.axvline(0.5, color='k', ls=':', lw=1, label='chance')
ax.set_xlabel('per-unit accuracy'); ax.set_ylabel('units')
ax.legend(fontsize=7); ax.set_title('accuracy across units', fontsize=9)

ax = axes[1]
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center',
                color='white' if cm_norm[i, j] > 0.5 else 'black', fontsize=10)
ax.set_xticks([0, 1]); ax.set_xticklabels(['non-anch.', 'anchored'], fontsize=8)
ax.set_yticks([0, 1]); ax.set_yticklabels(['non-anch.', 'anchored'], fontsize=8)
ax.set_xlabel('inferred'); ax.set_ylabel('true')
ax.set_title('confusion (row-normalised)', fontsize=9)

ax = axes[2]
frac = np.nanmean(LAB[OK], axis=0)
ax.plot(frac, color='k', lw=1)
ax.axvspan(T1, T2, color='tab:red', alpha=0.12, label='cue absent (truth)')
ax.set_xlabel('trial'); ax.set_ylabel('fraction of units called anchored')
ax.set_ylim(0, 1); ax.legend(fontsize=7)
ax.set_title('population time course', fontsize=9)
for a in axes: a.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(FIGPATH, 'anchoring_validation_B_accuracy.pdf'),
            dpi=200, bbox_inches='tight')
plt.show()

print(f'sensitivity (anchored correctly called): {cm_norm[1,1]:.3f}')
print(f'specificity (non-anchored correctly called): {cm_norm[0,0]:.3f}')

## Panel C — does the phase-aliasing median filter earn its place?

The filter exists to remove isolated false-positive *anchored* labels inside runs of
genuinely non-anchored trials, which arise when a drifting periodic cell happens to
align in phase with other non-anchored trials. If it works, specificity should improve
with little cost to sensitivity.

In [ ]:
LAB_NF, ACC_NF, SIL_NF, OK_NF = classify_all(TC, TRUTH, medfilt_size=0)
both = OK & OK_NF
cm_nf = confusion_matrix(np.tile(TRUTH, int(both.sum())),
                         LAB_NF[both].ravel().astype(int), labels=[0, 1])
cm_nf = cm_nf / cm_nf.sum(axis=1, keepdims=True)
cm_f = confusion_matrix(np.tile(TRUTH, int(both.sum())),
                        LAB[both].ravel().astype(int), labels=[0, 1])
cm_f = cm_f / cm_f.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.2))
ax = axes[0]
ax.scatter(ACC_NF[both], ACC[both], s=4, alpha=0.25, color='0.35')
lim = [min(ACC_NF[both].min(), ACC[both].min()) - 0.02, 1.01]
ax.plot(lim, lim, ls=':', color='k', lw=1)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('accuracy, no median filter'); ax.set_ylabel('accuracy, median filter (size 5)')
ax.set_title(f'improved in {int((ACC[both] > ACC_NF[both]).sum())}/{int(both.sum())} units',
             fontsize=9)

ax = axes[1]
x = np.arange(2); w = 0.35
ax.bar(x - w/2, [cm_nf[0, 0], cm_nf[1, 1]], w, label='no filter', color='0.7')
ax.bar(x + w/2, [cm_f[0, 0], cm_f[1, 1]], w, label='median filter', color='#4a2a7a')
ax.set_xticks(x); ax.set_xticklabels(['specificity\n(non-anchored)', 'sensitivity\n(anchored)'],
                                      fontsize=8)
ax.set_ylim(0, 1); ax.legend(fontsize=7); ax.set_ylabel('fraction correct')
ax.set_title('where the filter acts', fontsize=9)
for a in axes: a.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(FIGPATH, 'anchoring_validation_C_medfilt.pdf'),
            dpi=200, bbox_inches='tight')
plt.show()

print(f'median accuracy: no filter {np.nanmedian(ACC_NF[both]):.3f} -> '
      f'filter {np.nanmedian(ACC[both]):.3f}')
print(f'specificity {cm_nf[0,0]:.3f} -> {cm_f[0,0]:.3f} | '
      f'sensitivity {cm_nf[1,1]:.3f} -> {cm_f[1,1]:.3f}')

## Panel D — how brief an epoch can it resolve?

The argument for preferring this classifier over the spectral one is temporal
resolution, so the honest floor on that should be measured rather than asserted.

Sessions of varying non-anchored epoch length are constructed from the cached
simulation by splicing the **first** *k* trials of the C block between the two A
blocks. Taking the first *k* is the principled choice: during C the hidden state drifts
progressively, so the first *k* C trials are exactly what a *k*-trial cue loss would
have produced. Splicing later C trials would sample a more-drifted state than a short
epoch ever reaches.

In [ ]:
A_pre = np.arange(0, T1)
C_all = np.arange(T1, T2)
A_post = np.arange(T2, N_TRIALS)

ks = [1, 2, 3, 4, 5, 7, 10, 15, 20, 25, 30]
rows = []
for k in ks:
    idx = np.concatenate([A_pre, C_all[:k], A_post])
    truth_k = np.concatenate([np.ones(len(A_pre)), np.zeros(k), np.ones(len(A_post))]).astype(int)
    tc_k = TC[:, idx, :]
    lab_k, acc_k, _, ok_k = classify_all(tc_k, truth_k, medfilt_size=5)
    # recall on the non-anchored epoch specifically -- the quantity that degrades
    recall = np.nanmean((lab_k[ok_k][:, len(A_pre):len(A_pre)+k] == 0).mean(axis=1))
    rows.append(dict(k=k, acc=np.nanmedian(acc_k), recall=recall, n=int(ok_k.sum())))
    print(f'  epoch {k:>2} trials: median accuracy {rows[-1]["acc"]:.3f}, '
          f'non-anchored recall {recall:.3f}')

import pandas as pd
D = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(D.k, D.recall, 'o-', color='#4a2a7a', lw=1.2, ms=4, label='non-anchored recall')
ax.plot(D.k, D.acc, 's--', color='0.5', lw=1, ms=3, label='overall accuracy')
ax.axhline(0.5, color='k', ls=':', lw=1)
ax.set_xlabel('length of non-anchored epoch (trials)')
ax.set_ylabel('fraction correct'); ax.set_ylim(0, 1.02)
ax.legend(fontsize=7); ax.spines[['top', 'right']].set_visible(False)
ax.set_title('resolution floor: how short an epoch is still detected?', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIGPATH, 'anchoring_validation_D_epoch_length.pdf'),
            dpi=200, bbox_inches='tight')
plt.show()

## Panel E — is the classifier general, or grid-specific?

This is the panel that tests the claim made in the Methods. The classifier was chosen
over the spectral classifier because it was purpose-built to be general rather than
tuned on grid cells. If accuracy tracks gridness, that claim needs softening.

Simulated units span a wide range of grid scores. Low-gridness RNN units are not
literally ramping cells, so this is a weaker test than recorded non-grid spatial cells
would provide — but it is the only one where ground truth exists, and a flat
accuracy-vs-gridness relationship is the minimum the generality claim requires.

In [ ]:
gs = GRID_SCORES[OK]; ac = ACC[OK]
qs = np.quantile(gs, np.linspace(0, 1, 6))
bins = np.digitize(gs, qs[1:-1])
fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))

ax = axes[0]
ax.scatter(gs, ac, s=4, alpha=0.2, color='0.4')
for b in range(5):
    m = bins == b
    if m.sum():
        ax.plot(np.median(gs[m]), np.median(ac[m]), 'o', color='#c04744', ms=6)
        ax.plot([np.median(gs[m])]*2, np.percentile(ac[m], [25, 75]), color='#c04744', lw=1.5)
ax.axhline(0.5, color='k', ls=':', lw=1)
ax.set_xlabel('grid score'); ax.set_ylabel('classifier accuracy')
ax.set_title('accuracy vs gridness (red = quintile median, IQR)', fontsize=9)

ax = axes[1]
labs = [f'Q{i+1}' for i in range(5)]
data = [ac[bins == b] for b in range(5)]
bp = ax.boxplot(data, labels=labs, showfliers=False, patch_artist=True, widths=0.6)
for patch in bp['boxes']:
    patch.set_facecolor('#5f9ea0')
ax.axhline(0.5, color='k', ls=':', lw=1)
ax.set_xlabel('grid score quintile (Q1 = least grid-like)')
ax.set_ylabel('classifier accuracy')
ax.set_title('accuracy by gridness quintile', fontsize=9)
for a in axes: a.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(FIGPATH, 'anchoring_validation_E_generality.pdf'),
            dpi=200, bbox_inches='tight')
plt.show()

from scipy.stats import spearmanr, kruskal
r, p = spearmanr(gs, ac)
print(f'Spearman(grid score, accuracy) = {r:+.3f}, p = {p:.3g}')
print(f'Kruskal across quintiles: H = {kruskal(*data)[0]:.1f}, p = {kruskal(*data)[1]:.3g}')
for i, d in enumerate(data):
    print(f'  Q{i+1}: n={len(d)}, median accuracy {np.median(d):.3f}, '
          f'grid score {np.median(gs[bins==i]):+.2f}')

## What the validation shows

**Headline.** Median accuracy **0.990** (IQR 0.960–1.000) across all 1024 units; 92.3%
score ≥ 0.90. Sensitivity 0.989, specificity 0.906 — misses are almost entirely
non-anchored trials called anchored, not the reverse.

**The median filter earns its place, and then some.** Specificity rises 0.823 → 0.906
and median accuracy 0.950 → 0.990 when it is applied. That is a real correction, not
cosmetic tidying, and it acts exactly where predicted: on false-positive *anchored*
labels inside non-anchored runs.

**But it sets a hard resolution floor of 3 trials.** This is the important result and it
was not previously documented. A size-5 median filter erases any run shorter than 3, so
short anchoring epochs are removed *by construction*:

| epoch length | 1 | 2 | 3 | 4 | 5 | 10 | 20 |
|---|---|---|---|---|---|---|---|
| non-anchored recall | 0.02 | 0.03 | 0.71 | 0.81 | 0.83 | 0.87 | 0.91 |

One- and two-trial epochs are essentially never detected. Detection appears abruptly at
3 trials and plateaus from about 5. So the classifier does **not** have per-trial
resolution in practice: it has per-trial *labelling* with a 3-trial minimum detectable
epoch. That is still 3–4× finer than the eleven-trial support of the spectral
classifier, so the temporal-resolution argument holds — but it should be stated as
"3 consecutive trials", not "per trial".

This is a genuine trade-off to state rather than bury: the filter buys ~8 points of
specificity at the cost of blindness to epochs shorter than 3 trials. If brief anchoring
events matter for a particular analysis, that analysis should be run with
`medfilt_size=0` and the lower specificity accepted.

**Generality holds, but is not perfectly flat.** Accuracy correlates with gridness —
Spearman r = +0.219, p = 1.5e-12 (Kruskal across quintiles p = 2.2e-9). The effect is
statistically clear at n = 1024 but small in magnitude: the least grid-like quintile
(median grid score −0.15) still reaches 0.970 median accuracy against 0.990 for the most
grid-like (+0.68). So the classifier is not meaningfully grid-dependent, and the
generality claim in the Methods stands — but it is a 2-point gradient, not zero, and
reporting it as exactly flat would be wrong.

**Caveat on Panel E.** Low-gridness RNN units are not ramping cells. This is the only
setting with ground truth, but it is a weaker test of generality than recorded non-grid
spatial cells would provide. The claim it supports is "accuracy does not depend
strongly on periodic structure", not "accuracy is identical for ramping cells".


## Summary

In [ ]:
print(f"""
ANCHORING CLASSIFIER VALIDATION (RNN ground truth, A -> C -> A cue-loss session)

  units classified          {int(OK.sum())}/{N_UNITS}
  median accuracy           {np.nanmedian(ACC):.3f}   (IQR {np.nanpercentile(ACC,25):.3f}-{np.nanpercentile(ACC,75):.3f})
  units >= 0.90 accuracy    {int((ACC>=0.90).sum())} ({100*np.nanmean(ACC>=0.90):.1f}%)
  sensitivity / specificity {cm_norm[1,1]:.3f} / {cm_norm[0,0]:.3f}
  median filter             {np.nanmedian(ACC_NF[both]):.3f} -> {np.nanmedian(ACC[both]):.3f} median accuracy
  gridness dependence       Spearman r = {spearmanr(gs, ac)[0]:+.3f} (p = {spearmanr(gs, ac)[1]:.3g})

Panels written to {FIGPATH}:
  anchoring_validation_A_examples.pdf     worked examples, true vs inferred
  anchoring_validation_B_accuracy.pdf     accuracy, confusion, population time course
  anchoring_validation_C_medfilt.pdf      phase-aliasing filter on/off
  anchoring_validation_D_epoch_length.pdf resolution floor
  anchoring_validation_E_generality.pdf   accuracy vs gridness
""")